# Scenario 1: Late deliveries and customer satisfaction

Business question: when an order arrives after the promised date, does the
customer rate it worse, and by how much?

Run from the project root:  python analysis/01_delivery.py
Reads:  data/cleaned/orders_clean.csv, reviews_dedup.csv, order_items_agg.csv, customers_clean.csv
Writes: outputs/exports/delivery_*.csv  and  outputs/figures/delivery_*.png

In [1]:
import polars as pl
from scipy import stats
import matplotlib
matplotlib.use("Agg")                     # draw to files, no window
import matplotlib.pyplot as plt

In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(14)

polars.config.Config

## 1. Load
`try_parse_dates=True` makes Polars read the timestamp columns as real
Datetime instead of text, so we can subtract them later.

In [3]:
orders = pl.read_csv("data/cleaned/orders_clean.csv", try_parse_dates=True)
reviews = pl.read_csv("data/cleaned/reviews_dedup.csv", try_parse_dates=True)
items = pl.read_csv("data/cleaned/order_items_agg.csv")
customers = pl.read_csv("data/cleaned/customers_clean.csv",
                        schema_overrides={"customer_zip_code_prefix": pl.String})

In [4]:
orders.shape, reviews.shape, items.shape, customers.shape

((99441, 14), (98673, 7), (98666, 5), (99441, 5))

In [5]:
orders.schema

Schema([('order_id', String),
        ('customer_id', String),
        ('order_status', String),
        ('order_purchase_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('order_approved_at', Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_carrier_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_customer_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_estimated_delivery_date', Date),
        ('flag_delivered_but_no_date', Boolean),
        ('flag_carrier_before_approved', Boolean),
        ('flag_customer_before_carrier', Boolean),
        ('valid_for_delay_calc', Boolean),
        ('days_late', Float64),
        ('delivery_status', String)])

## 2. Keep only orders we can judge
Three filters, all decided during cleaning:
- status must be `delivered` (a shipped or canceled order has no delivery date to compare)
- `valid_for_delay_calc` must be True (this drops the 1390 rows whose timestamps contradict each other)
- the delivery date must exist (8 delivered orders have none)

In [6]:
orders["order_status"].value_counts().sort("count", descending=True)

order_status,count
str,u32
"""delivered""",96478
"""shipped""",1107
"""canceled""",625
"""unavailable""",609
"""invoiced""",314
"""processing""",301
"""created""",5
"""approved""",2


In [7]:
delivered = orders.filter(
    (pl.col("order_status") == "delivered")
    & pl.col("valid_for_delay_calc")
    & pl.col("order_delivered_customer_date").is_not_null()
)
delivered.shape                          # 95,097 orders we can judge

(95097, 14)

## 3. Attach the review and the order value
`reviews_dedup` has exactly one row per order, so this join cannot multiply rows.
`how="left"` keeps orders with no review; we count them, then drop them for the
score analysis only.

In [8]:
delivered = delivered.join(reviews.select("order_id", "review_score"), on="order_id", how="left")
delivered = delivered.join(items.select("order_id", "item_price_total", "freight_value_total"), on="order_id", how="left")
delivered.shape                          # same row count as before the join

(95097, 17)

In [9]:
delivered.filter(pl.col("review_score").is_null()).height    # orders with no review at all

639

In [10]:
judged = delivered.filter(pl.col("review_score").is_not_null())
judged.shape

(94458, 17)

## 4. The headline: late vs on time
`delivery_status` and `days_late` were computed in cleaning:
late means the order arrived on a later calendar date than the estimate.

In [11]:
summary = (
    judged.group_by("delivery_status")
    .agg(
        pl.len().alias("orders"),
        pl.col("review_score").mean().round(3).alias("mean_score"),
        (pl.col("review_score") <= 2).mean().round(4).alias("share_1_2_star"),
        (pl.col("review_score") == 1).mean().round(4).alias("share_1_star"),
        pl.col("item_price_total").sum().round(2).alias("revenue"),
    )
    .sort("delivery_status")
)
summary

delivery_status,orders,mean_score,share_1_2_star,share_1_star,revenue
str,u32,f64,f64,f64,f64
"""late""",6357,2.269,0.6248,0.538,949935.07
"""on_time""",88101,4.29,0.0926,0.0662,1.1984e7


In [12]:
late_rate = judged.filter(pl.col("delivery_status") == "late").height / judged.height
round(late_rate * 100, 2)               # % of judged orders that arrived late

6.73

## 5. Is the gap real, and how big is it?
Welch's t-test compares the two means without assuming equal spread.
The confidence interval is the number to put on the slide: the gap in stars,
with the range it could plausibly sit in.
With ~95k orders almost any gap is "significant"; the interval is what tells
you whether it is big enough to act on.

In [13]:
late_scores = judged.filter(pl.col("delivery_status") == "late")["review_score"].to_numpy()
ontime_scores = judged.filter(pl.col("delivery_status") == "on_time")["review_score"].to_numpy()
len(late_scores), len(ontime_scores)

(6357, 88101)

In [14]:
test = stats.ttest_ind(ontime_scores, late_scores, equal_var=False)
ci = test.confidence_interval(confidence_level=0.95)
gap = float(ontime_scores.mean() - late_scores.mean())
round(gap, 3), (round(ci.low, 3), round(ci.high, 3)), test.pvalue

(2.021, (np.float64(1.981), np.float64(2.06)), np.float64(0.0))

## 6. Does it get worse the later it is?
Bucket the delay to show a dose-response. If score keeps falling as delay
grows, the relationship is not a fluke of one cutoff.

In [15]:
judged = judged.with_columns(
    pl.when(pl.col("days_late") <= 0).then(pl.lit("on time"))
    .when(pl.col("days_late") <= 3).then(pl.lit("1-3 days late"))
    .when(pl.col("days_late") <= 7).then(pl.lit("4-7 days late"))
    .when(pl.col("days_late") <= 14).then(pl.lit("8-14 days late"))
    .otherwise(pl.lit("15+ days late"))
    .alias("delay_bucket")
)

In [16]:
bucket_order = ["on time", "1-3 days late", "4-7 days late", "8-14 days late", "15+ days late"]
by_bucket = (
    judged.group_by("delay_bucket")
    .agg(
        pl.len().alias("orders"),
        pl.col("review_score").mean().round(3).alias("mean_score"),
        (pl.col("review_score") <= 2).mean().round(4).alias("share_1_2_star"),
    )
    .with_columns(pl.col("delay_bucket").replace_strict(bucket_order, list(range(5))).alias("order_key"))
    .sort("order_key")
    .drop("order_key")
)
by_bucket

delay_bucket,orders,mean_score,share_1_2_star
str,u32,f64,f64
"""on time""",88101,4.29,0.0926
"""1-3 days late""",1842,3.288,0.3225
"""4-7 days late""",1741,2.105,0.6766
"""8-14 days late""",1444,1.672,0.8012
"""15+ days late""",1330,1.723,0.7842


## 7. Where is it happening? Late rate by customer state
Always carry the order count. A state with 40 orders and a state with
40,000 should not sit side by side as if they were equally certain.

In [17]:
judged = judged.join(customers.select("customer_id", "customer_state"), on="customer_id", how="left")

In [18]:
by_state = (
    judged.group_by("customer_state")
    .agg(
        pl.len().alias("orders"),
        (pl.col("delivery_status") == "late").mean().round(4).alias("late_rate"),
        pl.col("review_score").mean().round(3).alias("mean_score"),
        pl.col("days_late").filter(pl.col("delivery_status") == "late").mean().round(1).alias("avg_days_late_when_late"),
    )
    .sort("late_rate", descending=True)
)
by_state

customer_state,orders,late_rate,mean_score,avg_days_late_when_late
str,u32,f64,f64,f64
"""AL""",390,0.2051,3.867,9.1
"""MA""",700,0.1743,3.836,10.5
"""SE""",332,0.1506,3.907,16.4
"""PI""",464,0.1401,4.002,13.5
"""CE""",1254,0.1388,3.943,15.0
"""RR""",40,0.125,3.875,36.4
"""RJ""",12074,0.1203,3.963,13.5
"""BA""",3183,0.1191,3.927,11.9
"""PA""",923,0.1062,3.913,13.1


## 8. When is it happening? Late rate by purchase month
If lateness spikes in specific months, the recommendation is about capacity
planning, not about carriers.

In [19]:
by_month = (
    judged.with_columns(pl.col("order_purchase_timestamp").dt.strftime("%Y-%m").alias("month"))
    .group_by("month")
    .agg(
        pl.len().alias("orders"),
        (pl.col("delivery_status") == "late").mean().round(4).alias("late_rate"),
        pl.col("review_score").mean().round(3).alias("mean_score"),
    )
    .sort("month")
    .filter(pl.col("orders") >= 100)        # drop the first and last partial months
)
by_month

month,orders,late_rate,mean_score
str,u32,f64,f64
"""2016-10""",258,0.0078,4.019
"""2017-01""",740,0.0284,4.199
"""2017-02""",1640,0.0256,4.204
"""2017-03""",2526,0.0447,4.188
"""2017-04""",2266,0.0644,4.136
"""2017-05""",3502,0.0283,4.235
"""2017-06""",3106,0.0293,4.223
"""2017-07""",3811,0.0276,4.262
"""2017-08""",4162,0.0291,4.314


## 9. Business impact
Two numbers for the slide.
- Revenue that arrived late: money tied to a bad experience.
- "Excess" 1-2 star reviews: how many fewer bad reviews there would be if late
  orders had the on-time bad-review rate. This is a what-if, not a measurement,
  and the slide should say so.

In [20]:
late = judged.filter(pl.col("delivery_status") == "late")
late_revenue = late["item_price_total"].sum()
ontime_bad_rate = (judged.filter(pl.col("delivery_status") == "on_time")["review_score"] <= 2).mean()
late_bad_actual = (late["review_score"] <= 2).sum()
late_bad_expected = late.height * ontime_bad_rate
excess_bad_reviews = late_bad_actual - late_bad_expected
round(late_revenue, 2), late_bad_actual, round(late_bad_expected), round(excess_bad_reviews)

(949935.07, 3972, 589, 3383)

In [21]:
impact = pl.DataFrame({
    "metric": [
        "judged_orders", "late_orders", "late_rate",
        "mean_score_on_time", "mean_score_late", "score_gap", "score_gap_ci_low", "score_gap_ci_high",
        "share_1_2_star_on_time", "share_1_2_star_late",
        "late_revenue_brl", "excess_bad_reviews_from_lateness",
    ],
    "value": [
        float(judged.height), float(late.height), round(late_rate, 4),
        round(float(ontime_scores.mean()), 3), round(float(late_scores.mean()), 3),
        round(gap, 3), round(float(ci.low), 3), round(float(ci.high), 3),
        round(float(ontime_bad_rate), 4), round(float((late["review_score"] <= 2).mean()), 4),
        round(float(late_revenue), 2), float(round(float(excess_bad_reviews))),
    ],
})
impact                                   # one column of numbers, all Float64 so the table has one type

metric,value
str,f64
"""judged_orders""",94458.0
"""late_orders""",6357.0
"""late_rate""",0.0673
"""mean_score_on_…",4.29
"""mean_score_lat…",2.269
"""score_gap""",2.021
"""score_gap_ci_l…",1.981
"""score_gap_ci_h…",2.06
"""share_1_2_star…",0.0926


## 10. Export for the dashboard
Column names here are the contract with the dashboard. Do not rename them.

In [22]:
impact.write_csv("outputs/exports/delivery_summary.csv")
by_bucket.write_csv("outputs/exports/delivery_by_bucket.csv")
by_state.write_csv("outputs/exports/delivery_by_state.csv")
by_month.write_csv("outputs/exports/delivery_by_month.csv")

## 11. Figures
matplotlib takes plain lists, so each chart is: pull two columns out of a
Polars table with `.to_list()`, hand them to a bar or line call, save.

In [23]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(by_bucket["delay_bucket"].to_list(), by_bucket["mean_score"].to_list(), color="#4C72B0")
ax.set_ylim(1, 5)
ax.set_ylabel("Mean review score (1 to 5)")
ax.set_title("Review score falls as delivery delay grows")
for i, v in enumerate(by_bucket["mean_score"].to_list()):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.savefig("outputs/figures/delivery_score_by_delay_bucket.png", dpi=150)
plt.close()

In [24]:
top_states = by_state.head(15)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_states["customer_state"].to_list(), (top_states["late_rate"] * 100).to_list(), color="#DD8452")
ax.invert_yaxis()
ax.set_xlabel("Late delivery rate (%)")
ax.set_title("Late delivery rate, 15 worst states (label shows order count)")
for i, (rate, n) in enumerate(zip(top_states["late_rate"].to_list(), top_states["orders"].to_list())):
    ax.text(rate * 100 + 0.3, i, f"n={n:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("outputs/figures/delivery_late_rate_by_state.png", dpi=150)
plt.close()

In [25]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(by_month["month"].to_list(), (by_month["late_rate"] * 100).to_list(), marker="o", color="#C44E52")
ax.set_ylabel("Late delivery rate (%)")
ax.set_title("Late delivery rate by purchase month")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("outputs/figures/delivery_late_rate_by_month.png", dpi=150)
plt.close()

In [26]:
print("Scenario 1 done.")
print(f"  judged orders: {judged.height:,}   late: {late.height:,} ({late_rate*100:.1f}%)")
print(f"  mean score on time {ontime_scores.mean():.2f} vs late {late_scores.mean():.2f}  gap {gap:.2f} stars  95% CI [{ci.low:.2f}, {ci.high:.2f}]")
print(f"  revenue delivered late: R$ {late_revenue:,.0f}   excess 1-2 star reviews: {excess_bad_reviews:,.0f}")

Scenario 1 done.
  judged orders: 94,458   late: 6,357 (6.7%)
  mean score on time 4.29 vs late 2.27  gap 2.02 stars  95% CI [1.98, 2.06]
  revenue delivered late: R$ 949,935   excess 1-2 star reviews: 3,383
